<a href="https://colab.research.google.com/github/Sherlysukmadira/2311532015_Sherly-Sukmadira-Putri_ImageProcessing/blob/main/cleaning_dirty_financial_transactions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



# ***`PEMBERSIHAN DATA`***


' data pada umumnya masih berbentuk mentah, pembersihan data di perlukan sehingga menjadikan data lebih tingga peforma untuk tahapan lanjutan '

' pada kali ini kami menggunakan salah satu data finansial pada suatu market place, kami membersihkan data dengan tujuan menghasilkan data yang bersih sehingga data finansial yang sebelumnya kotor ( data duplikat, tidak relevan, hilang ) menjadi data yang bersih '

Langkah-langkah pembersihan data yang kami lakukan meliputi:
1. Tampilkan Data & Visualisasikan (Exploratory Data Analysis - EDA)
2. Cek Tipe Data & Sesuaikan   
3. Identifikasi & Buang Data Duplikat
4. Identifikasi & Tangani Data Hilang (Missing Values)
5. Identifikasi & Buang Data yang Tidak Relevan
6. Identifikasi Outlier & Tangani
7. Perbaiki Kesalahan Struktural & Standarisasi Format
8. Visualisasi Data Pasca-Pembersihan

# ***IMPORT LIBRARIES YANG DIPERLUKAN***

In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, StandardScaler

# ***BACA DATA DENGAN LIBRARY PANDAS***

Data yang digunakan dibaca menggunakan library pandas dan dijadikan variabel data. ini akan memudahkan dalam pembacaan dan pemanggilan data pada tahap selanjutnya

In [ ]:
data = pd.read_csv('dirty_financial_transactions.csv')

# ***TAMPILKAN TIPE DATA***
Penggunaan perintah `info` sebagai pengecekan type data pada masing masing features, seperti terlihat  type data dari features data ini berupa object yang meliputi dari transaction_id, Transaction_date, Costumer_id, Product_name, Price, payment_Method, dan Transaction_status. selain itu terdapat tipe data float64 pada feature/kolom Quantity

In [ ]:
print(data.info())

# ***PENGUJIAN PEMANGGILAN DATA***
Ini dilakukan untuk membaca data per baris penggunaan `head` yang memanggil nilai teratas pada data dan `tail` nilai terbawah pada data

In [ ]:
print(data.head(10))

In [ ]:
print(data.tail(10))

# ***STATISTIK DATA FLOAT***
Dikarenakan terdapat salah satu data bertipe float maka statistik diperlukan untuk melihat kemungkinan yang bsia dilakukan pada data tersebut, pada kasus pembersihan jika terdapat outlier atau missing value maka statistik ini dapat menangani hal tersebut. Penggunaan perintah `describe` akan memunculkan statistik tersebut

In [ ]:
print(data.describe())

# ***KESELURUHAN DATA***

Keseluruhan data ditampilkan untuk memudahkan dalam navigasi data, penggunaan library pandas digunakan dengan perintah `set_option` . Output dikeluarkan dengan `display max row`yang mengeluarkan output semua baris, `display.max.columns` mengeluarkan output semua kolom, `display.width`mengeluarkan output sehingga compatible dengan layar digunakan, `display.max_coldwidth`mengeluarkan output pada baris dikolom tanpa terpotong. Sehingga dengan ini data ditampilkan sesuai dengan data mentah digunakan

In [ ]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

data


# ***VISUALISASIKAN DATA***

Keseluruhan dari data terlihat pada tabel sebelumnya, dengan ini visualisasi dengan berbagai diagram akan membantu dalam menccari korelasi hubungan masing masing variabel

In [ ]:
# prompt: visualisasikan data

import matplotlib.pyplot as plt
import seaborn as sns

# Visualisasi distribusi harga
plt.figure(figsize=(10, 6))
sns.histplot(data['Price'], kde=True)
plt.title('Distribusi Harga')
plt.xlabel('Harga')
plt.ylabel('Frekuensi')
plt.show()

# Visualisasi jumlah transaksi per metode pembayaran
plt.figure(figsize=(10, 6))
sns.countplot(x='payment_Method', data=data)
plt.title('Jumlah Transaksi per Metode Pembayaran')
plt.xlabel('Metode Pembayaran')
plt.ylabel('Jumlah Transaksi')
plt.xticks(rotation=45)
plt.show()

# Visualisasi hubungan antara harga dan kuantitas
plt.figure(figsize=(10, 6))
sns.scatterplot(x='Price', y='Quantity', data=data)
plt.title('Hubungan antara Harga dan Kuantitas')
plt.xlabel('Harga')
plt.ylabel('Kuantitas')
plt.show()

# Matriks korelasi (jika ada kolom numerik lainnya)
numeric_cols = data.select_dtypes(include=np.number).columns
if len(numeric_cols) > 1:
  plt.figure(figsize=(10, 8))
  sns.heatmap(data[numeric_cols].corr(), annot=True, cmap='coolwarm')
  plt.title('Matriks Korelasi')
  plt.show()

# Contoh visualisasi lain (sesuaikan dengan kebutuhan)
# ...


# ***LAKUKAN PENYESUAIAN FORMAT TANGGAL***

Pada dataset ini terdapat kesalahan dalam format tanggal. Maka diperlukan penanganan dengan membersihkan dan menstandarkan format tanggal dalam dataset agar sesuai dengan tipe data `datetime64[ns]`.

In [ ]:
# Function to clean and standardize dates
def clean_date(date_str):
    if pd.isna(date_str):
        return np.nan  # Keep NaN values
    try:
        return pd.to_datetime(date_str, errors='coerce')  # Try automatic parsing
    except:
        match = re.search(r'(\d{4})-(\d{2})-(\d{2})', str(date_str))
        if match:
            try:
                return pd.to_datetime(f"{match.group(1)}-{match.group(2)}-{match.group(3)}", errors='coerce')
            except:
                return pd.to_datetime('2025-02-28')  # Default if correction fails
        return np.nan  # Invalid format, mark as NaN

# Ensure column is string before applying
data['Transaction_Date'] = data['Transaction_Date'].astype(str).apply(clean_date)

# Remove invalid dates (NaN)
data.dropna(subset=['Transaction_Date'], inplace=True)

# Convert to datetime64[ns]
data['Transaction_Date'] = pd.to_datetime(data['Transaction_Date'])

# Verify the results
print(data.info())
print(data.head())


Fungsi `clean_date()` pertama-tama memeriksa apakah nilai yang diberikan adalah `NaN` (hilang), jika iya, maka akan tetap dibiarkan sebagai `NaN`. Kemudian, fungsi mencoba mengonversi nilai tersebut menggunakan `pd.to_datetime()` dengan `errors='coerce'`, yang secara otomatis akan mengubah nilai yang tidak valid menjadi `NaT` tanpa menyebabkan error. Jika konversi langsung gagal, program menggunakan ekspresi reguler (`re.search`) untuk mencari pola tanggal dalam format `YYYY-MM-DD`. Jika pola ini ditemukan, fungsi akan mencoba mengonversinya lagi. Jika masih gagal, nilai default `2025-02-28` digunakan sebagai pengganti. Setelah fungsi diterapkan pada kolom `Transaction_Date`, baris dengan nilai `NaN` akan dihapus, dan akhirnya seluruh kolom dikonversi ke format datetime untuk memastikan keseragaman data. Program ini memastikan bahwa data tanggal dalam dataset lebih bersih dan siap untuk analisis lebih lanjut.

# ***MENANGANI NILAI NEGATIF PADA KOLOM QUANTITY DAN PRICE***

## **Kolom Quantity**

In [ ]:
# Create a new column 'Is_Return' and initialize it to False
data['Is_Return'] = data['Quantity'] < 0

# Convert all negative quantities to positive
data['Quantity'] = data['Quantity'].abs()

# Display the updated DataFrame
print(data.head())


## **Kolom Price**

In [ ]:
# Convert 'Price' column to float, handling errors
data['Price'] = pd.to_numeric(data['Price'], errors='coerce').astype(float)

# Create a new column 'Is_Refund' and initialize it to False
data['Is_Refund'] = data['Price'] < 0

# Convert all negative prices to positive
data['Price'] = data['Price'].abs()

# Display the updated DataFrame
print(data.info())
print(data.head())

In [ ]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

data


# ***MEMERIKSA DUPLIKASI DATA***

Untuk mendeteksi dan menghapus transaksi duplikat dapat dilihat berdasarkan kolom `Transaction_ID`. Pertama, program mencari duplikasi dengan menggunakan fungsi `duplicated()`, yang menandai setiap transaksi dengan `Transaction_ID` yang muncul lebih dari satu kali. Jika ditemukan duplikat, program akan mencetak daftar transaksi yang memiliki ID yang sama. Setelah itu, program menghapus semua duplikasi tetapi tetap menyimpan **satu transaksi pertama** yang muncul menggunakan `drop_duplicates(keep='first')`. Setelah proses penghapusan, program menampilkan DataFrame yang telah diperbarui dan melakukan verifikasi dengan `info()` serta `head()` untuk memastikan bahwa pembersihan telah berhasil. Dengan pendekatan ini, program memastikan bahwa setiap transaksi memiliki ID yang unik, menghindari inkonsistensi dalam analisis data.

In [ ]:
# Check for duplicate transactions based on 'Transaction_ID'
duplicate_transactions = data[data.duplicated(subset=['Transaction_ID'], keep=False)]

# If there are duplicates
if not duplicate_transactions.empty:
    print("Duplicate transactions found:")
    print(duplicate_transactions)

    # Remove duplicates, keeping the first occurrence
    data = data.drop_duplicates(subset=['Transaction_ID'], keep='first')

    print("\nDuplicates removed. Updated DataFrame:")
    print(data)
else:
    print("No duplicate transactions found.")

# Verify the results
print(data.info())
print(data.head())


# **INDENTIFIKASI DATA YANG HILANG DAN OUTLIER**

In [ ]:
data = pd.read_csv("dirty_financial_transactions.csv")
missing_values = data.isnull().sum()
print("Missing values per column:\n", missing_values)

# Analyze missing values in more detail (optional)
# Example: Percentage of missing values in each column
missing_percentage = (data.isnull().sum() / len(data)) * 100
print("\nMissing value percentage per column:\n", missing_percentage)


Hapus data Transaction_ID  

In [ ]:
data_cleaned = data.dropna(subset=["Transaction_ID"])


Isi data Transaction_Date

In [ ]:

data = pd.read_csv('dirty_financial_transactions.csv')


In [ ]:
feature = "Transaction_Date"

plt.figure(figsize=(12, 5))

# Histogram
plt.subplot(1, 2, 1)
sns.histplot(data[feature], bins=30, kde=True)
plt.title(f"Histogram of {feature}")

# Boxplot
plt.subplot(1, 2, 2)
sns.boxplot(x=data[feature])
plt.title(f"Boxplot of {feature}")

plt.show()

MENANGANI OUTLIER

In [ ]:
data['Transaction_Date'] = pd.to_datetime(data['Transaction_Date'], errors='coerce')

Q1 = data['Transaction_Date'].quantile(0.25)
Q3 = data['Transaction_Date'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Hapus outlier
data_cleaned = data[(data['Transaction_Date'] >= lower_bound) & (data['Transaction_Date'] <= upper_bound)]


In [ ]:
data['Transaction_Date'].isnull().sum()


In [ ]:
data['Transaction_Date'] = pd.to_datetime(data['Transaction_Date'], errors='coerce')


In [ ]:
data['Transaction_Date'].fillna(data['Transaction_Date'].mode()[0], inplace=True)  # Pakai mode (nilai paling sering muncul)


In [ ]:
data['Transaction_Date'].isnull().sum()

HAPUS DATA KOSONG CUSTOMER_ID


In [ ]:
data['Customer_ID'].isnull().sum()

In [ ]:
data.dropna(subset=['Customer_ID'], inplace=True) #Hapus missing value Customer_ID


In [ ]:
data['Customer_ID'].isnull().sum()

ISI DATA QUANTITI

In [ ]:
feature = "Quantity"

plt.figure(figsize=(12, 5))

# Histogram
plt.subplot(1, 2, 1)
sns.histplot(data[feature], bins=30, kde=True)
plt.title(f"Histogram of {feature}")

# Boxplot
plt.subplot(1, 2, 2)
sns.boxplot(x=data[feature])
plt.title(f"Boxplot of {feature}")

plt.show()

Histogram miring ke kanan jadi kita gunakan median untuk mengisi data yang kosong


MENANGANI OUTLIER


In [ ]:
# Pastikan Price dalam bentuk numerik
data['Quantity'] = pd.to_numeric(data['Quantity'], errors='coerce')

# Hitung IQR
Q1 = data['Quantity'].quantile(0.25)
Q3 = data['Quantity'].quantile(0.75)
IQR = Q3 - Q1

# Tentukan batas bawah dan atas untuk outlier
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Hapus outlier
data_cleaned = data[(data['Quantity'] >= lower_bound) & (data['Quantity'] <= upper_bound)]

# Cek jumlah data setelah membersihkan outlier
print(f"Jumlah data sebelum: {len(data)}")
print(f"Jumlah data setelah menghapus outlier: {len(data_cleaned)}")


In [ ]:
data["Quantity"].fillna(data["Quantity"].median(), inplace=True)


In [ ]:
data['Quantity'].isnull().sum()

ISI DATA PRICE

In [ ]:
feature = "Price"

plt.figure(figsize=(12, 5))

# Histogram
plt.subplot(1, 2, 1)
sns.histplot(data[feature], bins=30, kde=True)
plt.title(f"Histogram of {feature}")

# Boxplot
plt.subplot(1, 2, 2)
sns.boxplot(x=data[feature])
plt.title(f"Boxplot of {feature}")

plt.show()

MENANGANI OUTLIER PRICE

In [ ]:
# Pastikan Price dalam bentuk numerik
data['Price'] = pd.to_numeric(data['Price'], errors='coerce')

# Hitung IQR
Q1 = data['Price'].quantile(0.25)
Q3 = data['Price'].quantile(0.75)
IQR = Q3 - Q1

# Tentukan batas bawah dan atas untuk outlier
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Hapus outlier
data_cleaned = data[(data['Price'] >= lower_bound) & (data['Price'] <= upper_bound)]

# Cek jumlah data setelah membersihkan outlier
print(f"Jumlah data sebelum: {len(data)}")
print(f"Jumlah data setelah menghapus outlier: {len(data_cleaned)}")


In [ ]:
data['Price'] = data['Price'].replace('[\$]', '', regex=True).astype(float)
#The replace pattern now includes parentheses
print(data['Price'].skew())

Nilai skewness 0.0049 menunjukkan bahwa distribusi hampir simetris (mendekati 0). Untuk analisis lebih lanjut, mean bisa digunakan sebagai representasi pusat data, karena tidak terlalu terpengaruh oleh outlier

In [ ]:
data["Price"].fillna(data["Price"].mean(), inplace=True)

In [ ]:
data['Price'].isnull().sum()

ISI DATA TRANSACTION STATUS

In [ ]:
feature = "Transaction_Status"

plt.figure(figsize=(12, 5))

# Histogram
plt.subplot(1, 2, 1)
sns.histplot(data[feature], bins=30, kde=True)
plt.title(f"Histogram of {feature}")

# Boxplot
plt.subplot(1, 2, 2)
sns.boxplot(x=data[feature])
plt.title(f"Boxplot of {feature}")

plt.show()

Dari grafik histogram, terlihat bahwa Transaction_Status memiliki beberapa kategori dengan variasi penulisan:

completed, Completed, complete → kemungkinan seharusnya sama.
Jadi, sebelum mengisi data kosong, kita perlu membersihkan data kategori terlebih dahulu.

In [ ]:
data['Transaction_Status'] = data['Transaction_Status'].str.lower()
data['Transaction_Status'] = data['Transaction_Status'].replace({'complete': 'completed'})


In [ ]:
feature = "Transaction_Status"

plt.figure(figsize=(12, 5))

# Histogram
plt.subplot(1, 2, 1)
sns.histplot(data[feature], bins=30, kde=True)
plt.title(f"Histogram of {feature}")

# Boxplot
plt.subplot(1, 2, 2)
sns.boxplot(x=data[feature])
plt.title(f"Boxplot of {feature}")

plt.show()

MENANGANI OUTLIER TRANSACTION STATUS

Karna completed Trsansaction status merupakan data yang terbanyak maka kita isi missing valuenya dengan completed

In [ ]:
data['Transaction_Status'].fillna(data['Transaction_Status'].mode()[0], inplace=True)


In [ ]:
data['Transaction_Status'].isnull().sum()

In [ ]:
# prompt: setelah semua proses ini saya ingin data bersih dijadikan nama file baru clean_data_finansial tanpa menghilangkan data file sebelumnya

# Save the cleaned data to a new CSV file
data.to_csv('clean_data_finansial.csv', index=False)


# ***STANDARISASI FORMAT DAN INCONSISTENCY DATA***

Perbaikan kesalahan struktural dan standarisasi format dilakukan agar data menjadi lebih konsisten. Kesalahan seperti typo atau inkonsistensi dalam penulisan kategori bisa menyebabkan kesalahan dalam analisis, misalnya jika ada perbedaan dalam penulisan jenis kelamin seperti "Male" dan "male". Selain itu, format angka, tanggal, dan teks perlu diseragamkan agar data lebih mudah dikelola dan digunakan.

In [ ]:
# Standardize column names to lowercase
data.columns = data.columns.str.lower()

# Further standardization of specific columns

# Example: Standardize 'product_name' (assuming some variations exist)
data['product_name'] = data['product_name'].str.lower()  # Convert to lowercase
data['product_name'] = data['product_name'].str.strip() # Remove leading/trailing spaces
# Add more specific standardization steps for 'product_name' if needed

# Example: Standardize 'payment_method'
data['payment_method'] = data['payment_method'].str.lower() #Convert to lower case
data['payment_method'] = data['payment_method'].str.strip() #Remove leading/trailing spaces
# ... handle other inconsistencies like typos or different spellings


# Example: Handling inconsistencies in 'customer_id' (if any exist)
# Assuming 'customer_id' should only contain numbers
# You might use regex to check and clean non-numeric characters
data['customer_id'] = data['customer_id'].astype(str).str.replace(r'\D+', '', regex=True) #Remove any non-digit char

# Convert numerical columns to appropriate types
for col in ['quantity', 'price']:
    data[col] = pd.to_numeric(data[col], errors='coerce')  # Convert to numeric, handling errors


# Verify the results after standardization
print(data.info())
print(data.head())

# Save the cleaned and standardized data
data.to_csv('standardized_financial_transactions.csv', index=False)

# ***VISUALISASI DATA SETELAH DIBERSIHKAN***

In [ ]:
# prompt: kemudian visualisasikan data yang sudah bersih dan distandarisasi itu (standardized_financial_transactions.csv)

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load the standardized data
data = pd.read_csv('standardized_financial_transactions.csv')

# Example visualizations

# 1. Distribution of transaction amounts
plt.figure(figsize=(8, 6))
sns.histplot(data['price'], kde=True)
plt.title('Distribution of Transaction Amounts')
plt.xlabel('Transaction Amount')
plt.ylabel('Frequency')
plt.show()

# 2. Transaction volume over time
plt.figure(figsize=(10, 6))
data['transaction_date'] = pd.to_datetime(data['transaction_date'])
data.groupby(data['transaction_date'].dt.date)['transaction_id'].count().plot()
plt.title('Transaction Volume Over Time')
plt.xlabel('Date')
plt.ylabel('Number of Transactions')
plt.show()

# 3. Number of transactions per payment method
plt.figure(figsize=(8, 6))
sns.countplot(x='payment_method', data=data)
plt.title('Number of Transactions per Payment Method')
plt.xlabel('Payment Method')
plt.ylabel('Number of Transactions')
plt.xticks(rotation=45, ha='right')  # Rotate x-axis labels for better readability
plt.show()

# 4. Transaction status distribution
plt.figure(figsize=(8, 6))
sns.countplot(x='transaction_status', data=data)
plt.title('Transaction Status Distribution')
plt.xlabel('Transaction Status')
plt.ylabel('Count')
plt.show()

# Add more visualizations as needed to explore other aspects of the data